In [12]:
import os

print("Các Dataset đang được gắn:")
for name in os.listdir("/kaggle/input"):
    print("-", name)

Các Dataset đang được gắn:
- datasets


# ***NGHIỆM THU 17/8***

In [13]:
import os

root = "/kaggle/input"

for current, dirs, files in os.walk(root):
    depth = current[len(root):].count(os.sep)

    if depth > 4:
        dirs[:] = []
        continue

    indent = "  " * depth
    print(f"{indent}{os.path.basename(current)}/")

input/
  datasets/
    ojanek/
      system-th/
        system thử/
      aic2026-siglip2/
        aic2026_siglip2/
      aic2026-siglip2-so400m/
        aic2026_siglip2_so400m/
      aic2026-data-nhe/
        aic2026_data_nhe/


In [14]:
from pathlib import Path
import shutil

source = Path("/kaggle/input/datasets/ojanek/system-th/system thử")
destination = Path("/kaggle/working/system_thu")

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Đã sao chép:", destination)
print("Có config:", (destination / "config_kaggle.yaml").exists())
print("Có file chấm điểm:", (destination / "do_diem_thuc_te.py").exists())
print("Có tests:", (destination / "tests").is_dir())

Đã sao chép: /kaggle/working/system_thu
Có config: True
Có file chấm điểm: True
Có tests: True


In [15]:
%cd /kaggle/working/system_thu

!pip install -q sentencepiece

/kaggle/working/system_thu


In [18]:
from pathlib import Path

config_file = Path("config_kaggle.yaml")
content = config_file.read_text(encoding="utf-8")

replacements = {
    "/kaggle/input/aic2026-data-nhe/aic2026_data_nhe":
        "/kaggle/input/datasets/ojanek/aic2026-data-nhe/aic2026_data_nhe",

    "/kaggle/input/aic2026-siglip2-so400m/aic2026_siglip2_so400m":
        "/kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m",

    "/kaggle/input/aic2026-siglip2-base/aic2026_siglip2":
        "/kaggle/input/datasets/ojanek/aic2026-siglip2/aic2026_siglip2",
}

for old, new in replacements.items():
    content = content.replace(old, new)

config_file.write_text(content, encoding="utf-8")
print("Đã sửa config_kaggle.yaml")

Đã sửa config_kaggle.yaml


In [19]:
paths = [
    "/kaggle/input/datasets/ojanek/aic2026-data-nhe/aic2026_data_nhe",
    "/kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m",
    "/kaggle/input/datasets/ojanek/aic2026-siglip2/aic2026_siglip2",
]

for path in paths:
    print(Path(path).exists(), path)

True /kaggle/input/datasets/ojanek/aic2026-data-nhe/aic2026_data_nhe
True /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
True /kaggle/input/datasets/ojanek/aic2026-siglip2/aic2026_siglip2


In [20]:
!python -X utf8 -m pytest -q

........................................................................ [ 24%]
........................................................................ [ 49%]
........................................................................ [ 74%]
........................................................................ [ 98%]
...                                                                      [100%]
=============================== warnings summary ===============================
tests/test_cau_hinh.py::test_dap_an_qa_ra_tieng_viet_khi_config_noi_vi
  <frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute

tests/test_cau_hinh.py::test_dap_an_qa_ra_tieng_viet_khi_config_noi_vi
  <frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
291 passed, 2 warnings in 25.65s
sys:1: DeprecationWarning: builtin type swigvarl

In [22]:
!python -X utf8 do_diem_thuc_te.py --config=config_kaggle.yaml

Đo theo CẤU HÌNH THI:
  config          : /kaggle/working/system_thu/config_kaggle.yaml
  kế thừa từ     : /kaggle/working/system_thu/config.example.yaml
  encoder         : siglip2 google/siglip2-so400m-patch14-384
  vector ảnh      : /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
  đáp án QA       : vi (dịch: opus_mt)
  QA VLM          : gemini_vlm/gemini-3.6-flash -> cloudflare_vlm/@cf/meta/llama-3.2-11b-vision-instruct
  vlm_top_n       : 10 · loại câu: ['color', 'open']
  portfolio       : tắt
  ngôn ngữ đề     : en (bản dùng để truy hồi)
  số dòng/truy vấn: 100

  177,321 event / 873 video
  hợp nhất: 1 encoder phụ (trọng số chính 2)
  encoder: SigLIP2TextEncoder google/siglip2-so400m-patch14-384 (1152 chiều) · cache 210 câu

Dev set: 170 câu
config.json: 100%|█████████████████████████████| 559/559 [00:00<00:00, 1.70MB/s]
tokenizer_config.json: 47.2kB [00:00, 78.9MB/s]
tokenizer.json: 100%|██████████████████████| 34.4M/34.4M [00:00<00:00, 54.0MB/s]
to

# ***BƯỚC 1***

In [23]:
import os
import re
from pathlib import Path

input_root = Path("/kaggle/input")
video_pattern = re.compile(r"L\d+_V\d+", re.IGNORECASE)
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

image_roots = []

for current, directories, files in os.walk(input_root):
    current_path = Path(current)

    video_directories = [
        name for name in directories
        if video_pattern.fullmatch(name)
    ]

    if not video_directories:
        continue

    # Kiểm tra một thư mục video mẫu có chứa ảnh thật không
    sample_video = current_path / video_directories[0]

    try:
        contains_images = any(
            file.is_file() and file.suffix.lower() in image_extensions
            for file in sample_video.iterdir()
        )
    except PermissionError:
        contains_images = False

    if contains_images:
        image_roots.append(current_path)
        print(
            "✓ Tìm thấy:",
            current_path,
            f"({len(video_directories)} thư mục video)"
        )

        # Không cần đi sâu vào hàng trăm thư mục ảnh
        directories[:] = []

✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes (25 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes (88 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes (31 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes (100 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes (23 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes (100 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes (24 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_a/keyframes (99 thư mục video)
✓ Tìm thấy: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_e/keyframes (99 thư mục video)


In [24]:
print("Số image root tìm được:", len(image_roots))

for root in image_roots:
    video_dirs = sorted([
        path for path in root.iterdir()
        if path.is_dir() and video_pattern.fullmatch(path.name)
    ])

    print("\nROOT:", root)
    print("Số thư mục video:", len(video_dirs))
    print("Ví dụ video:", [p.name for p in video_dirs[:3]])

    if video_dirs:
        sample_images = sorted([
            p.name for p in video_dirs[0].iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions
        ])
        print("Ví dụ ảnh:", sample_images[:5])

Số image root tìm được: 14

ROOT: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes
Số thư mục video: 25
Ví dụ video: ['L23_V001', 'L23_V002', 'L23_V003']
Ví dụ ảnh: ['001.jpg', '002.jpg', '003.jpg', '004.jpg', '005.jpg']

ROOT: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes
Số thư mục video: 88
Ví dụ video: ['L25_V001', 'L25_V002', 'L25_V003']
Ví dụ ảnh: ['001.jpg', '002.jpg', '003.jpg', '004.jpg', '005.jpg']

ROOT: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes
Số thư mục video: 31
Ví dụ video: ['L22_V001', 'L22_V002', 'L22_V003']
Ví dụ ảnh: ['001.jpg', '002.jpg', '003.jpg', '004.jpg', '005.jpg']

ROOT: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes
Số thư mục video: 100
Ví dụ video: ['L26_V100', 'L26_V101', 'L26_V102']
Ví dụ ảnh: ['001.jpg', '002.jpg', '003.jpg', '004.jpg', '005.jpg']

ROOT: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L2

In [25]:
from pathlib import Path
import yaml

config_path = Path("/kaggle/working/system_thu/config_kaggle.yaml")

config = yaml.safe_load(
    config_path.read_text(encoding="utf-8")
)

config.setdefault("qa", {})
config["qa"]["image_roots"] = [
    str(path) for path in image_roots
]

config_path.write_text(
    yaml.safe_dump(
        config,
        allow_unicode=True,
        sort_keys=False
    ),
    encoding="utf-8"
)

print("Đã ghi image_roots:")
for path in config["qa"]["image_roots"]:
    print("-", path)

Đã ghi image_roots:
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_a/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_e/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_d/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L21/keyframes
- /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L24/keyframe

In [26]:
from aic2026 import config as CFG

config = CFG.load_config("config_kaggle.yaml")

print("Image roots đang dùng:")
for root in config["qa"]["image_roots"]:
    path = Path(root)
    print(
        "✓" if path.is_dir() else "✗",
        root
    )

Image roots đang dùng:
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_a/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_e/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_d/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L21/keyframes
✓ /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L24/keyfr

In [29]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

required_keys = [
    "GEMINI_API_KEY",
    "CLOUDFLARE_API_TOKEN",
    "CLOUDFLARE_ACCOUNT_ID",
]

for key in required_keys:
    try:
        os.environ[key] = secrets.get_secret(key)
        print("✓ Đã nạp", key)
    except Exception as error:
        print("✗ Không nạp được", key, "-", type(error).__name__)

✓ Đã nạp GEMINI_API_KEY
✓ Đã nạp CLOUDFLARE_API_TOKEN
✓ Đã nạp CLOUDFLARE_ACCOUNT_ID


In [33]:
!pip install -q --upgrade "google-genai>=2.0.0,<3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.8 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.56.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.

In [34]:
from importlib.metadata import version

print("google-genai:", version("google-genai"))

google-genai: 2.18.1


In [35]:
!python -X utf8 kiem_khoa_api.py


Ảnh thử: 001_L22_V006_n209.jpg  ·  câu hỏi: 'Trong ảnh có người không?'

1. gemini_vlm / gemini-3.6-flash
   ✅ 4.35s · đáp án: 'có' · biến môi trường: GEMINI_API_KEY, GOOGLE_API_KEY
2. cloudflare_vlm / @cf/meta/llama-3.2-11b-vision-instruct
   ✅ 0.64s · đáp án: 'Có' · biến môi trường: CLOUDFLARE_API_TOKEN, CLOUDFLARE_ACCOUNT_ID

  2/2 provider trả lời được.


In [37]:
from pathlib import Path
from aic2026 import config as CFG

config = CFG.load_config("config_kaggle.yaml")
roots = config.get("qa", {}).get("image_roots", [])

print("Image roots:")
for root in roots:
    print(
        "✓ tồn tại" if Path(root).is_dir() else "✗ không tồn tại",
        root
    )

Image roots:
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_a/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_e/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_d/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L21/keyframe

In [39]:
!python -X utf8 kiem_resolver_anh.py --config=config_kaggle.yaml --so 120

Thư mục ảnh:
  /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes  (25 thư mục video, vd ['L23_V001', 'L23_V002', 'L23_V003'])
  /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes  (88 thư mục video, vd ['L25_V001', 'L25_V002', 'L25_V003'])
  /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes  (31 thư mục video, vd ['L22_V001', 'L22_V002', 'L22_V003'])
  /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes  (100 thư mục video, vd ['L26_V100', 'L26_V101', 'L26_V102'])
  /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes  (23 thư mục video, vd ['L29_V001', 'L29_V002', 'L29_V003'])
  /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes  (100 thư mục video, vd ['L26_V200', 'L26_V201', 'L26_V202'])
  /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes  (24 thư mục video, vd ['L28_V001', 'L28_V002', 

In [44]:
!python -X utf8 dien_tap_qa_anh.py --config=config_kaggle.yaml --so 3

Thư mục ảnh: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_a/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_e/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_d/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L21/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L24/keyframes, /kaggle/input/dat

In [45]:
!python -X utf8 dien_tap_qa_anh.py --config=config_kaggle.yaml 

Thư mục ảnh: /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_a/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_e/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_d/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L21/keyframes, /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L24/keyframes, /kaggle/input/dat

# ***BƯỚC B2***

In [46]:
import json
from pathlib import Path

de_bay = {
    "queries": [
        {
            "qid": "kis-1",
            "task": "kis",

            # Cố tình là tiếng Việt không dấu
            "text": "Mot nguoi nong dan dang lai may keo",

            # Hệ thống phải chọn khóa này
            "text_en": "A farmer is driving a tractor through a field"
        },
        {
            "qid": "kis-2",
            "task": "kis",

            # Cố tình chỉ có tiếng Việt để kiểm tra cảnh báo
            "text": "Một người đang chuẩn bị thức ăn trong nhà bếp"
        },
        {
            "qid": "qa-1",
            "task": "qa",

            "text_en": "A person wearing a red shirt is speaking indoors",
            "text_vi": "Một người mặc áo đỏ đang phát biểu trong nhà",

            "question_en": "What color is the person's shirt?",
            "_cau_hoi_vi": "Áo của người đang phát biểu có màu gì?"
        },
        {
            "qid": "trake-1",
            "task": "trake",

            # Đây là một chuỗi; hệ thống phải tách thành đúng 3 sự kiện
            "sub_events": (
                "a man sits across from an officer"
                " → the man signs some documents"
                " → the man stands up and leaves"
            )
        }
    ]
}

path = Path("/kaggle/working/system_thu/de_bay.json")

path.write_text(
    json.dumps(
        de_bay,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("✓ Đã tạo:", path)
print(path.read_text(encoding="utf-8"))

✓ Đã tạo: /kaggle/working/system_thu/de_bay.json
{
  "queries": [
    {
      "qid": "kis-1",
      "task": "kis",
      "text": "Mot nguoi nong dan dang lai may keo",
      "text_en": "A farmer is driving a tractor through a field"
    },
    {
      "qid": "kis-2",
      "task": "kis",
      "text": "Một người đang chuẩn bị thức ăn trong nhà bếp"
    },
    {
      "qid": "qa-1",
      "task": "qa",
      "text_en": "A person wearing a red shirt is speaking indoors",
      "text_vi": "Một người mặc áo đỏ đang phát biểu trong nhà",
      "question_en": "What color is the person's shirt?",
      "_cau_hoi_vi": "Áo của người đang phát biểu có màu gì?"
    },
    {
      "qid": "trake-1",
      "task": "trake",
      "sub_events": "a man sits across from an officer → the man signs some documents → the man stands up and leaves"
    }
  ]
}


In [47]:
!python -X utf8 nop_bai.py \
    --config=config_kaggle.yaml \
    --truy-van=de_bay.json \
    --ra=out/de_bay.zip

  NỘP BÀI — CẤU HÌNH ĐANG DÙNG
  config          : /kaggle/working/system_thu/config_kaggle.yaml
  kế thừa từ     : /kaggle/working/system_thu/config.example.yaml
  encoder         : siglip2 google/siglip2-so400m-patch14-384
  vector ảnh      : /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
  đáp án QA       : vi (dịch: opus_mt)
  QA VLM          : gemini_vlm/gemini-3.6-flash -> cloudflare_vlm/@cf/meta/llama-3.2-11b-vision-instruct
  vlm_top_n       : 10 · loại câu: ['color', 'open']
  portfolio       : tắt
  ngôn ngữ đề     : en (bản dùng để truy hồi)
  số dòng/truy vấn: 100

ĐỀ: de_bay.json  ·  chính sách ngôn ngữ `en`
  ✓ kis-1          en  (khoá 'text_en')
  ⚠ kis-2          vi  (khoá 'text')
  ✓ qa-1           en  (khoá 'text_en')
  ✓ trake-1        en  (khoá 'sub_events')

  ⚠ 1/4 câu KHÔNG theo được chính sách `en`:
     kis-2: chính sách `en` nhưng đề chỉ có bản `vi` (khoá 'text') — đang dùng bản đó.  Đo trên dev set 120 câu KIS (SigLIP2 base-224): 

# ***BƯỚC B3***

In [50]:
!python -X utf8 do_diem_thuc_te.py --config=config_kaggle.yaml --vi

Đo theo CẤU HÌNH THI:  [TRUY VẤN TIẾNG VIỆT]
  config          : /kaggle/working/system_thu/config_kaggle.yaml
  kế thừa từ     : /kaggle/working/system_thu/config.example.yaml
  encoder         : siglip2 google/siglip2-so400m-patch14-384
  vector ảnh      : /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
  đáp án QA       : vi (dịch: opus_mt)
  QA VLM          : gemini_vlm/gemini-3.6-flash -> cloudflare_vlm/@cf/meta/llama-3.2-11b-vision-instruct
  vlm_top_n       : 10 · loại câu: ['color', 'open']
  portfolio       : tắt
  ngôn ngữ đề     : en (bản dùng để truy hồi)
  số dòng/truy vấn: 100

  177,321 event / 873 video
  hợp nhất: 1 encoder phụ (trọng số chính 2)
  encoder: SigLIP2TextEncoder google/siglip2-so400m-patch14-384 (1152 chiều) · cache 210 câu

Dev set: 170 câu  (truy vấn TIẾNG VIỆT, không qua lớp dịch)
Loading weights: 100%|█| 888/888 [00:00<00:00, 1320.45it/s, Materializing param=
Loading weights: 100%|█| 408/408 [00:00<00:00, 1378.88it/s, Mater

# ***BƯỚC 4***

In [51]:
from pathlib import Path

runbook = Path("/kaggle/working/system_thu/QA_VLM_RUNBOOK.md")

start_marker = "<!-- B4_KAGGLE_2026_START -->"
end_marker = "<!-- B4_KAGGLE_2026_END -->"

b4_content = f"""
{start_marker}

## Nghiệm thu Kaggle B0–B4 — 17/08/2026

### Môi trường đã nghiệm thu

- Dataset: 873 video / 177.321 keyframe.
- Encoder chính: `google/siglip2-so400m-patch14-384`, 1152 chiều.
- Encoder phụ: SigLIP2 base-224, hợp nhất RRF với trọng số encoder chính = 2.
- Portfolio: tắt.
- Chính sách ngôn ngữ truy vấn: ưu tiên `en`.
- Số dòng nộp tối đa: 100 dòng/truy vấn.
- Bộ kiểm thử: 291 test xanh.

### B0 — Đường tiếng Anh chính

| Chỉ số | Giá trị |
|---|---:|
| Điểm gộp core | 0.3122 |
| KIS | 0.3750 |
| QA cheap | 0.0867 |
| TRAKE | 0.2733 |
| Recall@1 | 11,8% |
| Recall@100 | 44,7% |

Kết luận: mốc KIS 0.3750 được tái lập đúng trên Kaggle. Cấu hình dữ liệu,
so400m và encoder phụ đều hoạt động đúng.

### B1 — Đường ảnh QA

- Corpus keyframe BTC đã được mount qua `qa.image_roots`.
- Resolver ảnh đã được diễn tập trên Kaggle.
- Gemini là provider chính.
- Cloudflare Workers AI là provider dự phòng.
- Kaggle Secrets cần nạp vào `os.environ` sau mỗi lần restart session.
- Kaggle cần `google-genai >= 2.0.0` để dùng Interactions API.

### B2 — Đề song ngữ và TRAKE

- Hệ thống ưu tiên khóa `text_en` khi đề có cả hai bản.
- Câu chỉ có tiếng Việt được cảnh báo rõ ràng.
- Chuỗi TRAKE dùng dấu `→` được tách đúng thành các sự kiện.
- File diễn tập tạo đủ 100 dòng mỗi truy vấn.
- ZIP nghiệm thu: `ok=True`, không có lỗi.
- Đáp án QA xuất ra bằng tiếng Việt.

### B3 — Đường tiếng Việt trực tiếp

| Chỉ số | Giá trị |
|---|---:|
| Điểm gộp core | 0.2463 |
| KIS | 0.2883 |
| QA cheap | 0.0867 |
| TRAKE | 0.2333 |
| Recall@1 | 8,2% |
| Recall@100 | 35,3% |

So với tiếng Anh:

- KIS: 0.3750 → 0.2883, giảm 0.0867, khoảng 23%.
- TRAKE: 0.2733 → 0.2333, giảm 0.0400, khoảng 15%.
- Recall@100: 44,7% → 35,3%.

Quyết định: giữ tiếng Anh làm đường truy hồi chính. Tiếng Việt trực tiếp là
đường dự phòng khi truy vấn không có bản tiếng Anh.

Không bật portfolio dựa trên riêng phép đo tiếng Việt; cần validation split độc
lập trước khi đổi mặc định.

### Quy trình batch 2

BTC thông báo dữ liệu vòng sơ tuyển đầy đủ sẽ có thêm batch 2.

Khi batch 2 xuất hiện:

1. Gắn video/keyframe/map-keyframes batch 2 vào Kaggle.
2. Encode ảnh batch 2 bằng SigLIP2 so400m cho encoder chính.
3. Encode lại batch 2 bằng SigLIP2 base-224 cho encoder phụ.
4. Không trộn vector của hai checkpoint vào cùng thư mục.
5. Cập nhật đường dẫn Dataset trong `config_kaggle.yaml`.
6. Kiểm tra số video và số vector ảnh khớp nhau.
7. Chạy lại 291 test.
8. Chạy lại B0 và yêu cầu KIS không giảm bất thường.
9. Chạy lại `kiem_resolver_anh.py` với corpus ảnh đầy đủ.
10. Diễn tập `nop_bai.py` và yêu cầu ZIP `ok=True`.

Dành tối thiểu 5 giờ GPU trong quota 30 giờ/tuần để encode cả so400m và base
cho batch 2.

### Quyết định cuối

- Đường thi chính: tiếng Anh + so400m×2 + base-224.
- Đường dự phòng: tiếng Việt trực tiếp.
- QA chính: Gemini VLM.
- QA dự phòng: Cloudflare VLM.
- Portfolio: tắt.
- Không thay model hoặc tinh chỉnh thêm trước khi có batch 2/validation mới.

{end_marker}
""".strip()

old_content = (
    runbook.read_text(encoding="utf-8")
    if runbook.exists()
    else "# QA VLM Runbook\n"
)

if start_marker in old_content and end_marker in old_content:
    before = old_content.split(start_marker, 1)[0].rstrip()
    after = old_content.split(end_marker, 1)[1].lstrip()

    new_content = (
        before
        + "\n\n"
        + b4_content
        + ("\n\n" + after if after else "")
    )
else:
    new_content = old_content.rstrip() + "\n\n" + b4_content + "\n"

runbook.write_text(new_content, encoding="utf-8")

print("✅ Đã ghi B4 vào:", runbook)

✅ Đã ghi B4 vào: /kaggle/working/system_thu/QA_VLM_RUNBOOK.md


In [52]:
content = runbook.read_text(encoding="utf-8")
b4_section = content.split(start_marker, 1)[1].split(end_marker, 1)[0]

print(b4_section)



## Nghiệm thu Kaggle B0–B4 — 17/08/2026

### Môi trường đã nghiệm thu

- Dataset: 873 video / 177.321 keyframe.
- Encoder chính: `google/siglip2-so400m-patch14-384`, 1152 chiều.
- Encoder phụ: SigLIP2 base-224, hợp nhất RRF với trọng số encoder chính = 2.
- Portfolio: tắt.
- Chính sách ngôn ngữ truy vấn: ưu tiên `en`.
- Số dòng nộp tối đa: 100 dòng/truy vấn.
- Bộ kiểm thử: 291 test xanh.

### B0 — Đường tiếng Anh chính

| Chỉ số | Giá trị |
|---|---:|
| Điểm gộp core | 0.3122 |
| KIS | 0.3750 |
| QA cheap | 0.0867 |
| TRAKE | 0.2733 |
| Recall@1 | 11,8% |
| Recall@100 | 44,7% |

Kết luận: mốc KIS 0.3750 được tái lập đúng trên Kaggle. Cấu hình dữ liệu,
so400m và encoder phụ đều hoạt động đúng.

### B1 — Đường ảnh QA

- Corpus keyframe BTC đã được mount qua `qa.image_roots`.
- Resolver ảnh đã được diễn tập trên Kaggle.
- Gemini là provider chính.
- Cloudflare Workers AI là provider dự phòng.
- Kaggle Secrets cần nạp vào `os.environ` sau mỗi lần restart session.
- Kaggle cần `google-gen

In [53]:
from pathlib import Path

summary_path = Path("/kaggle/working/B4_NGHIEM_THU_AIC2026.md")
summary_path.write_text(
    "# B4 — Nghiệm thu AIC 2026\n\n" + b4_section.strip() + "\n",
    encoding="utf-8"
)

print("✅ Đã tạo:", summary_path)
print("Dung lượng:", summary_path.stat().st_size, "bytes")

✅ Đã tạo: /kaggle/working/B4_NGHIEM_THU_AIC2026.md
Dung lượng: 3518 bytes


# ***ĐO QA THẬT ***

In [54]:
%cd /kaggle/working/system_thu

/kaggle/working/system_thu


In [56]:
import os
from pathlib import Path
from aic2026 import config as CFG

config_path = Path("config_kaggle.yaml").resolve()
config = CFG.load_config(config_path)

qa = config.get("qa", {})

print("CONFIG:", config_path)
print("answer_candidates:", qa.get("answer_candidates"))
print("tail_answer:", qa.get("tail_answer"))
print("vlm_top_n:", qa.get("vlm_top_n"))

print("\nImage roots:")
image_roots = qa.get("image_roots", [])

for root in image_roots:
    print(
        "✓ tồn tại" if Path(root).is_dir() else "✗ không tồn tại",
        root
    )

print("\nAPI keys:")
required_keys = [
    "GEMINI_API_KEY",
    "CLOUDFLARE_API_TOKEN",
    "CLOUDFLARE_ACCOUNT_ID",
]

for key in required_keys:
    print(
        "✓ đã nạp" if os.environ.get(key) else "✗ chưa nạp",
        key
    )

missing = CFG.thieu_khoa_api(config)

assert not missing, f"Thiếu khóa API: {missing}"
assert qa.get("answer_candidates") == "luat", (
    "answer_candidates chưa phải luat"
)
assert qa.get("tail_answer") == "vlm", (
    "tail_answer chưa phải vlm"
)
assert image_roots, "qa.image_roots đang rỗng"
assert all(
    Path(root).is_dir()
    for root in image_roots
), "Có image_root không tồn tại"

print("\n✅ Tiền kiểm đạt — có thể đo 30 câu QA")

CONFIG: /kaggle/working/system_thu/config_kaggle.yaml
answer_candidates: luat
tail_answer: vlm
vlm_top_n: 10

Image roots:
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L23/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L25/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L22/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_b/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L29/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_c/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L28/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_a/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Keyframes_L26_e/keyframes
✓ tồn tại /kaggle/input/datasets/nguyenminhtric/data-batch-1-aic/Key

In [57]:
import copy
import csv
import json
import math
import time
from pathlib import Path

from aic2026 import config as CFG
from aic2026.core import scoring
from aic2026.core.offline_eval import load_queries
from aic2026.core.providers import RuleBasedProvider


def mean_and_se(values):
    mean = sum(values) / len(values)

    if len(values) < 2:
        return mean, float("inf")

    variance = sum(
        (value - mean) ** 2
        for value in values
    ) / (len(values) - 1)

    return mean, math.sqrt(variance / len(values))


def localization_ceiling(rows, ground_truth):
    scores = [
        scoring.r_score_kis(row, ground_truth)
        for row in rows
    ]

    return sum(
        max(scores[:k], default=0.0)
        for k in scoring.DEFAULT_KS
    ) / len(scoring.DEFAULT_KS)


# ------------------------------------------------------------
# 1. Đọc đúng cấu hình Kaggle
# ------------------------------------------------------------

config = CFG.load_config("config_kaggle.yaml")
run_config = copy.deepcopy(config)

# Dùng cache mới để phép đo này không lấy đáp án cũ trên máy Windows.
fresh_cache = Path(
    "out/vlm_qa_cache_gemini_kaggle_luat_20260817.json"
).resolve()

llm_config = run_config.get("llm", {})

if llm_config.get("provider") == "failover_vlm":
    llm_config["primary"]["cache_path"] = str(fresh_cache)

    for fallback in llm_config.get("fallbacks", []):
        if fallback.get("provider") == "cloudflare_vlm":
            fallback["cache_path"] = str(
                Path(
                    "out/vlm_qa_cache_cloudflare_kaggle_20260817.json"
                ).resolve()
            )
else:
    llm_config["cache_path"] = str(fresh_cache)


# ------------------------------------------------------------
# 2. Nạp dữ liệu và dựng đúng retrieval cấu hình thi
# ------------------------------------------------------------

print("Đang nạp dữ liệu...")

start = time.perf_counter()

events, embeddings, metadata, report = CFG.nap_du_lieu(
    run_config
)

secondary_retrievers = CFG.nap_hop_nhat(
    run_config,
    events
)

encoder = CFG.dung_encoder(run_config)

# Pipeline này chỉ dựng 100 dòng retrieval.
# Không cho nó tự gọi VLM lần nữa.
pipeline = CFG.dung_pipeline(
    run_config,
    events,
    embeddings,
    encoder=encoder,
    llm=RuleBasedProvider(),
    retrieval_phu=secondary_retrievers,
)

resolver = CFG.dung_image_resolver(run_config)
provider = CFG.dung_llm(run_config)

print(
    f"✓ {report.events:,} event / "
    f"{report.videos_da_nap} video"
)

print(
    f"✓ Encoder: {encoder.MODEL_ID} "
    f"({encoder.dim} chiều)"
)

print(
    f"✓ Encoder phụ: {len(secondary_retrievers)}"
)

print(
    f"✓ Nạp xong trong "
    f"{time.perf_counter() - start:.1f}s"
)


# ------------------------------------------------------------
# 3. Đọc 30 câu QA và resolve ảnh ground-truth
# ------------------------------------------------------------

dev_path = Path("data/dev_set_tu_lam.json")

pairs = [
    (query, ground_truth)
    for query, ground_truth in load_queries(str(dev_path))
    if query.task == "qa"
]

raw_data = json.loads(
    dev_path.read_text(encoding="utf-8")
)

raw_by_qid = {
    item["qid"]: item
    for item in raw_data["queries"]
    if item["task"] == "qa"
}

records = []

for query, ground_truth in pairs:
    raw_query = raw_by_qid[query.qid]

    ordinal = int(
        raw_query["_ghi_chu"]["keyframe_n"]
    )

    event_id = (
        f"{ground_truth.video_id}#{ordinal:05d}"
    )

    image_path = resolver.resolve(event_id)

    question_vi = (
        query.question_vi
        or raw_query.get("_cau_hoi_vi")
        or query.text
    )

    records.append({
        "query": query,
        "ground_truth": ground_truth,
        "event_id": event_id,
        "image": image_path,
        "question_vi": question_vi,
    })

assert len(records) == 30, (
    f"Mong 30 câu QA, nhận {len(records)}"
)

print(
    f"\n✅ Resolve đủ {len(records)}/30 ảnh QA"
)


# ------------------------------------------------------------
# 4. Gọi Gemini thật
# ------------------------------------------------------------

requests = [
    (
        record["question_vi"],
        str(record["image"]),
        record["event_id"],
        "",
    )
    for record in records
]

before_cache = provider.cache_hits
before_network = provider.network_calls
before_backend = provider.backend_requests

print(
    "\nĐang gửi 30 câu QA tới Gemini..."
)

vlm_start = time.perf_counter()

answers = provider.answer_images(requests)

vlm_seconds = time.perf_counter() - vlm_start

used_provider = (
    getattr(provider, "last_provider", None)
    or type(provider).__name__
)

new_cache_hits = (
    provider.cache_hits - before_cache
)

new_answers = (
    provider.network_calls - before_network
)

backend_requests = (
    provider.backend_requests - before_backend
)

print("\nProvider thực tế:", used_provider)
print("Cache hit mới:", new_cache_hits)
print("Đáp án mới:", new_answers)
print("API request backend:", backend_requests)
print(f"Thời gian VLM: {vlm_seconds:.1f}s")

if "Gemini" not in used_provider:
    raise RuntimeError(
        "Provider thực tế không phải Gemini. "
        "Không dùng kết quả này làm điểm Gemini."
    )

assert len(answers) == 30


# ------------------------------------------------------------
# 5. Chấm điểm trên đúng 100 dòng retrieval
# ------------------------------------------------------------

scores = []
ceilings = []
answer_matches = 0
details = []

print(
    "\nĐang dựng và chấm 100 dòng cho từng câu..."
)

for record, answer in zip(records, answers):
    query = record["query"]
    ground_truth = record["ground_truth"]

    retrieval_start = time.perf_counter()

    rows = pipeline.answer(
        query,
        top_k=100
    )

    retrieval_seconds = (
        time.perf_counter() - retrieval_start
    )

    # Config đang dùng tail_answer: vlm:
    # đáp án Gemini được áp dụng cho cả 100 dòng.
    scored_rows = [
        dict(row, answer=answer)
        for row in rows
    ]

    score = scoring.final_score(
        scored_rows,
        ground_truth
    )

    ceiling = localization_ceiling(
        scored_rows,
        ground_truth
    )

    accepted_answers = [
        ground_truth.answer,
        *ground_truth.answer_alt,
    ]

    matched = any(
        scoring.answers_match(
            answer,
            accepted
        )
        for accepted in accepted_answers
    )

    scores.append(score)
    ceilings.append(ceiling)
    answer_matches += int(matched)

    details.append({
        "qid": query.qid,
        "event_id": record["event_id"],
        "image": str(record["image"]),
        "question_vi": record["question_vi"],
        "ground_truth": ground_truth.answer,
        "answer_alt": ";".join(
            ground_truth.answer_alt
        ),
        "prediction": answer,
        "answer_matches": int(matched),
        "final_score": f"{score:.6f}",
        "localization_ceiling": (
            f"{ceiling:.6f}"
        ),
        "retrieval_seconds": (
            f"{retrieval_seconds:.6f}"
        ),
        "provider": used_provider,
    })

    print(
        f"{query.qid}: "
        f"score={score:.2f} · "
        f"answer={answer!r}"
    )


# ------------------------------------------------------------
# 6. Tổng hợp và lưu kết quả
# ------------------------------------------------------------

mean_score, score_se = mean_and_se(scores)
mean_ceiling, ceiling_se = mean_and_se(
    ceilings
)

gaps = [
    ceiling - score
    for ceiling, score in zip(
        ceilings,
        scores,
    )
]

mean_gap, gap_se = mean_and_se(gaps)

output_path = Path(
    "out/vlm_qa_predictions_config_kaggle.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with output_path.open(
    "w",
    encoding="utf-8-sig",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(details[0]),
    )

    writer.writeheader()
    writer.writerows(details)


print("\n" + "=" * 70)
print("KẾT QUẢ QA VLM THẬT — CẤU HÌNH KAGGLE")
print("=" * 70)

print(
    f"QA VLM: {mean_score:.4f} "
    f"± {score_se:.4f}"
)

print(
    f"Trần định vị: "
    f"{mean_ceiling:.4f} "
    f"± {ceiling_se:.4f}"
)

print(
    f"Khoảng cách trần - thật: "
    f"{mean_gap:+.4f} "
    f"± {gap_se:.4f}"
)

print(
    f"Đáp án khớp nhãn: "
    f"{answer_matches}/30"
)

print(
    f"Provider thực tế: "
    f"{used_provider}"
)

print(
    f"Cache hit: {new_cache_hits} · "
    f"đáp án mới: {new_answers} · "
    f"backend requests: {backend_requests}"
)

print(
    f"Thời gian VLM: "
    f"{vlm_seconds:.1f}s"
)

print(
    f"Chi tiết CSV: {output_path}"
)

Đang nạp dữ liệu...
✓ 177,321 event / 873 video
✓ Encoder: google/siglip2-so400m-patch14-384 (1152 chiều)
✓ Encoder phụ: 1
✓ Nạp xong trong 32.6s

✅ Resolve đủ 30/30 ảnh QA

Đang gửi 30 câu QA tới Gemini...

Provider thực tế: GeminiVLMProvider
Cache hit mới: 0
Đáp án mới: 30
API request backend: 3
Thời gian VLM: 68.4s

Đang dựng và chấm 100 dòng cho từng câu...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

qa-001: score=0.80 · answer='xanh lá'
qa-002: score=0.00 · answer='trắng'
qa-003: score=0.60 · answer='tạp chí'
qa-004: score=0.20 · answer='nồi'
qa-005: score=0.00 · answer='muỗng'
qa-006: score=0.20 · answer='kính'
qa-007: score=0.00 · answer='2'
qa-008: score=0.00 · answer='đỏ'
qa-009: score=0.20 · answer='đỏ'
qa-010: score=0.00 · answer='trắng'
qa-011: score=0.00 · answer='2'
qa-012: score=0.00 · answer='trắng'
qa-013: score=0.00 · answer='cây đánh trứng'
qa-014: score=0.20 · answer='xe đạp'
qa-015: score=1.00 · answer='xanh dương'
qa-016: score=0.00 · answer='xanh dương'
qa-017: score=0.20 · answer='3'
qa-018: score=0.00 · answer='vá gỗ'
qa-019: score=0.00 · answer='xám'
qa-020: score=0.00 · answer='2'
qa-021: score=1.00 · answer='2'
qa-022: score=0.00 · answer='nâu'
qa-023: score=1.00 · answer='mì'
qa-024: score=0.00 · answer='hồng'
qa-025: score=0.00 · answer='vàng'
qa-026: score=0.20 · answer='xanh lá'
qa-027: score=0.80 · answer='đen'
qa-028: score=0.20 · answer='xám'
qa-029: 

In [1]:
import shutil
from pathlib import Path

nguon = Path("/kaggle/input/datasets/ojanek/system-th/system thử")
dich = Path("/kaggle/working/system_thu")

shutil.copytree(nguon, dich, dirs_exist_ok=True)
print("Đã sao chép code tới:", dich)

Đã sao chép code tới: /kaggle/working/system_thu


In [4]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ["GEMINI_API_KEY"] = secrets.get_secret("GEMINI_API_KEY")
os.environ["CLOUDFLARE_API_TOKEN"] = secrets.get_secret("CLOUDFLARE_API_TOKEN")
os.environ["CLOUDFLARE_ACCOUNT_ID"] = secrets.get_secret("CLOUDFLARE_ACCOUNT_ID")

print("✅ Đã nạp API key vào phiên Kaggle")

✅ Đã nạp API key vào phiên Kaggle


In [5]:
%cd /kaggle/working/system_thu
!python -X utf8 do_diem_qa_dau_cuoi.py --config config_kaggle.yaml

/kaggle/working/system_thu
  QA ĐẦU-CUỐI — pipeline tự gọi VLM trên khung nó tìm được
  config          : /kaggle/working/system_thu/config_kaggle.yaml
  kế thừa từ     : /kaggle/working/system_thu/config.example.yaml
  encoder         : siglip2 google/siglip2-so400m-patch14-384
  vector ảnh      : /kaggle/input/datasets/ojanek/aic2026-siglip2-so400m/aic2026_siglip2_so400m
  đáp án QA       : vi (dịch: opus_mt)
  QA VLM          : gemini_vlm/gemini-3.6-flash -> cloudflare_vlm/@cf/meta/llama-3.2-11b-vision-instruct
  vlm_top_n       : 10 · loại câu: ['color', 'open']
  portfolio       : tắt
  ngôn ngữ đề     : en (bản dùng để truy hồi)
  số dòng/truy vấn: 100

Đang nạp dữ liệu ...
  177,321 event / 873 video · hợp nhất 1 encoder phụ · 55.7s
  QA: FailoverVLMProvider · vlm_top_n 10 · tail_answer vlm · loại câu ['color', 'open']

Đo 30 câu QA ...

config.json: 100%|█████████████████████████████| 559/559 [00:00<00:00, 1.65MB/s]
tokenizer_config.json: 47.2kB [00:00, 66.5MB/s]
tokenizer.json